In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os



In [2]:
# Load splits
train_data = pd.read_pickle('dataset/train_data_final.pkl')
val_data_masked = pd.read_pickle('dataset/val_data_masked.pkl')
test_data_masked = pd.read_pickle('dataset/test_data_masked.pkl')

In [3]:
# Load ground truth
val_ground_truth = pd.read_pickle('dataset/val_ground_truth.pkl')
test_ground_truth = pd.read_pickle('dataset/test_ground_truth.pkl')

# Load mask indicators
val_mask_indicator = pd.read_pickle('dataset/val_mask_indices.pkl')
test_mask_indicator = pd.read_pickle('dataset/test_mask_indices.pkl')





In [4]:
print(f"\n✓ Loaded all data:")
print(f"  • Train: {train_data.shape}")
print(f"  • Val (masked): {val_data_masked.shape}")
print(f"  • Test (masked): {test_data_masked.shape}")


✓ Loaded all data:
  • Train: (143459, 49)
  • Val (masked): (30741, 49)
  • Test (masked): (30742, 49)


In [5]:
# Get features that were masked
features_to_impute = val_ground_truth.columns.tolist()
print(f"\n  • Features to impute: {len(features_to_impute)}")


  • Features to impute: 39


In [6]:
# Get all columns
all_columns = train_data.columns.tolist()

# Identify non-impute features
non_impute_features = [col for col in all_columns if col not in features_to_impute]

print(f"\n FEATURE SPLIT:")
print(f"  • Features to impute: {len(features_to_impute)}")
print(f"  • Features to keep as-is: {len(non_impute_features)}")


 FEATURE SPLIT:
  • Features to impute: 39
  • Features to keep as-is: 10


In [7]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

In [9]:

def clip_outliers_per_feature(data, lower_pct=0.5, upper_pct=99.5):
    """
    Clip outliers per feature based on percentiles.
    Handles NaN values properly.
    """
    data_clipped = data.copy()
    
    for i in range(data.shape[1]):
        col = data[:, i]
        
        # Get valid (non-NaN) values
        valid_mask = ~np.isnan(col)
        
        if valid_mask.sum() > 0:
            valid_values = col[valid_mask]
            
            # Calculate percentiles
            lower_bound = np.percentile(valid_values, lower_pct)
            upper_bound = np.percentile(valid_values, upper_pct)
            
            # Clip only valid values
            col[valid_mask] = np.clip(valid_values, lower_bound, upper_bound)
            data_clipped[:, i] = col
    
    return data_clipped

In [10]:

train_values = train_data[features_to_impute].values

In [11]:
train_values_clipped = clip_outliers_per_feature(train_values, lower_pct=0.5, upper_pct=99.5)


In [12]:
for i, feat in enumerate(features_to_impute[:10]):  # Show first 10
    original_max = np.nanmax(np.abs(train_values[:, i]))
    clipped_max = np.nanmax(np.abs(train_values_clipped[:, i]))
    
    if original_max > 1000:  # Only show features that were clipped significantly
        print(f"  • {feat[:45]:45s}: {original_max:>12.2e} → {clipped_max:>12.2e}")

In [13]:
scaler = StandardScaler()
scaler.fit(train_values_clipped)

# Transform all datasets (with outlier clipping)
train_scaled = scaler.transform(train_values_clipped)

val_values_clipped = clip_outliers_per_feature(val_data_masked[features_to_impute].values)
val_scaled = scaler.transform(val_values_clipped)

test_values_clipped = clip_outliers_per_feature(test_data_masked[features_to_impute].values)
test_scaled = scaler.transform(test_values_clipped)

In [14]:

new_stats = {
    'Min': np.nanmin(train_scaled),
    'Max': np.nanmax(train_scaled),
    'Mean': np.nanmean(train_scaled),
    'Std': np.nanstd(train_scaled),
    '1st percentile': np.nanpercentile(train_scaled, 1),
    '99th percentile': np.nanpercentile(train_scaled, 99)
}

print(f"{'Statistic':<20s} {'Value':<15s}")
print("─"*40)
for stat, value in new_stats.items():
    print(f"{stat:<20s} {value:<15.4f}")

print(f"\n✓ Values should now be in reasonable range (-5 to +5)")

# Check max values per feature
print(f"\nTop 5 features by max scaled value:")
feature_max_scaled = []
for i, feat in enumerate(features_to_impute):
    max_val = np.nanmax(np.abs(train_scaled[:, i]))
    feature_max_scaled.append((feat, max_val))

feature_max_scaled.sort(key=lambda x: x[1], reverse=True)

for feat, max_val in feature_max_scaled[:5]:
    print(f"  • {feat[:45]:45s}: {max_val:>10.4f}")


Statistic            Value          
────────────────────────────────────────
Min                  -3.9694        
Max                  8.8233         
Mean                 0.0000         
Std                  1.0000         
1st percentile       -2.0953        
99th percentile      2.2748         

✓ Values should now be in reasonable range (-5 to +5)

Top 5 features by max scaled value:
  • jitter                                       :     8.8233
  • Traffic Distance                             :     4.1352
  • Pos in Ref Round                             :     3.9694
  • PCell_Uplink_TB_Size                         :     3.9400
  • Altitude                                     :     3.8259


In [15]:
from sklearn.preprocessing import StandardScaler

# Fit scaler on training data
#scaler = StandardScaler()
#train_impute_array = train_data[features_to_impute].values
#scaler.fit(train_impute_array)


# Transform all sets
#train_scaled = scaler.transform(train_impute_array)
#val_scaled = scaler.transform(val_data_masked[features_to_impute].values)
#test_scaled = scaler.transform(test_data_masked[features_to_impute].values)



In [16]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer 
from sklearn.impute import KNNImputer, IterativeImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [17]:

def evaluate_imputation(imputed_data, ground_truth, mask_indicator, method_name, split_name):
  
    results = []
    all_errors_squared = []
    all_errors_abs = []
    
    for col in ground_truth.columns:
        # Get mask for this feature
        mask = mask_indicator[col]
        
        # Count masked values
        n_masked = mask.sum()
        
        if n_masked == 0:
            continue
        
        # Get ground truth values (only non-NaN)
        true_vals = ground_truth.loc[mask, col].dropna()
        
        
        # Get predicted values at same positions
        pred_vals = imputed_data.loc[true_vals.index, col]
        
        # Calculate errors
        errors = true_vals - pred_vals
        errors_squared = errors ** 2
        errors_abs = errors.abs()
        
        rmse = np.sqrt(errors_squared.mean())
        mae = errors_abs.mean()
        
        # Store results
        results.append({
            'feature': col,
            'n_masked': len(true_vals),
            'RMSE': rmse,
            'MAE': mae
        })
        
        # Collect for overall metrics
        all_errors_squared.extend(errors_squared.values)
        all_errors_abs.extend(errors_abs.values)
    
    # Overall metrics
    overall_rmse = np.sqrt(np.mean(all_errors_squared))
    overall_mae = np.mean(all_errors_abs)
    
    results_df = pd.DataFrame(results)
    
    return results_df, overall_rmse, overall_mae

print("\n EVALUATING METHODS...")



 EVALUATING METHODS...


In [18]:
def represent_columnwise_results(results_df, split_name="", method_name="", sort_by="RMSE", ascending=False, top_n=None):
    """
    Represent column-wise imputation performance in a clean table.
    """
    if results_df is None or results_df.empty:
        print("No column-wise results to display.")
        return results_df

    view = results_df.copy()

    # Ensure numeric formatting columns exist
    for c in ["n_masked", "RMSE", "MAE"]:
        if c in view.columns:
            view[c] = pd.to_numeric(view[c], errors="coerce")

    if sort_by in view.columns:
        view = view.sort_values(sort_by, ascending=ascending)

    if top_n is not None:
        view = view.head(top_n)

    title = f"\nColumn-wise Results"
    if split_name:
        title += f" | Split: {split_name}"
    if method_name:
        title += f" | Method: {method_name}"
    print(title)
    print("-" * len(title))

    print(
        view.to_string(
            index=False,
            formatters={
                "RMSE": lambda x: f"{x:.4f}" if pd.notna(x) else "nan",
                "MAE": lambda x: f"{x:.4f}" if pd.notna(x) else "nan",
            }
        )
    )

    return view


In [19]:
#forward/backward fill

def forward_backward_fill_imputation(data, features_to_impute):
    
    data_imputed = data.copy()
    
    # Apply forward fill then backward fill to impute features only
    for col in features_to_impute:
        if col in data_imputed.columns:
            # Forward fill
            data_imputed[col] = data_imputed[col].fillna(method='ffill')
            # Backward fill
            data_imputed[col] = data_imputed[col].fillna(method='bfill')
    
    return data_imputed

print("\n Applying Forward/Backward Fill...")

# Validation
print("\n  Processing validation set...")
val_ffill = forward_backward_fill_imputation(val_data_masked, features_to_impute)

# Test
print("  Processing test set...")
test_ffill = forward_backward_fill_imputation(test_data_masked, features_to_impute)




 Applying Forward/Backward Fill...

  Processing validation set...
  Processing test set...


C:\Users\Aditya\AppData\Local\Temp\ipykernel_14996\178238238.py:11: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data_imputed[col] = data_imputed[col].fillna(method='ffill')
C:\Users\Aditya\AppData\Local\Temp\ipykernel_14996\178238238.py:13: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data_imputed[col] = data_imputed[col].fillna(method='bfill')


In [20]:
# Check remaining missing
val_remaining = val_ffill[features_to_impute].isnull().sum().sum()
test_remaining = test_ffill[features_to_impute].isnull().sum().sum()

In [21]:
print(f"    Validation: {val_remaining:,} (should be 0 or very small)")
print(f"    Test: {test_remaining:,} (should be 0 or very small)")

    Validation: 0 (should be 0 or very small)
    Test: 0 (should be 0 or very small)


In [22]:
if val_remaining > 0 or test_remaining > 0:
    print(f"  Fill with column mean")
    
    # Fallback: fill with mean
    for col in features_to_impute:
        if val_ffill[col].isnull().sum() > 0:
            col_mean = train_data[col].mean()
            val_ffill[col].fillna(col_mean, inplace=True)
        
        if test_ffill[col].isnull().sum() > 0:
            col_mean = train_data[col].mean()
            test_ffill[col].fillna(col_mean, inplace=True)

In [23]:

method_name = 'Forward/Backward Fill'


# Validation
val_results, val_rmse, val_mae = evaluate_imputation(
    val_ffill, val_ground_truth, val_mask_indicator,
    method_name, 'Validation'
)

# Test
test_results, test_rmse, test_mae = evaluate_imputation(
    test_ffill, test_ground_truth, test_mask_indicator,
    method_name, 'Test'
)

print(f"\n  VALIDATION:")
print(f"    • Overall RMSE: {val_rmse:.4f}")
print(f"    • Overall MAE:  {val_mae:.4f}")
print(f"    • Features evaluated: {len(val_results)}")

print(f"\n  TEST:")
print(f"    • Overall RMSE: {test_rmse:.4f}")
print(f"    • Overall MAE:  {test_mae:.4f}")
print(f"    • Features evaluated: {len(test_results)}")
    


  VALIDATION:
    • Overall RMSE: 39849.6275
    • Overall MAE:  190.5827
    • Features evaluated: 39

  TEST:
    • Overall RMSE: 27849.2511
    • Overall MAE:  170.9081
    • Features evaluated: 39


In [24]:
val_results, val_rmse, val_mae = evaluate_imputation(
    val_ffill, val_ground_truth, val_mask_indicator, "Forward/Backward Fill", "Validation"
)

represent_columnwise_results(
    val_results,
    split_name="Validation",
    method_name="Forward/Backward Fill",
    sort_by="RMSE",
    ascending=False,
    top_n=50
)



Column-wise Results | Split: Validation | Method: Forward/Backward Fill
------------------------------------------------------------------------
                     feature  n_masked        RMSE       MAE
                      jitter      4514 249925.9846 5459.0661
      PCell_Uplink_frequency      4420   1079.3698  818.2233
    PCell_Downlink_frequency      4420   1077.9043  817.9432
              PCell_freq_MHz      4584    276.9702  171.0079
               PCell_Cell_ID      4420    167.4784  119.1853
                         COG      4611     66.1481   30.6038
 PCell_Uplink_Tx_Power_(dBm)      4564     38.8100   22.5445
            Traffic Distance      4611     24.3399    9.8294
              PCell_RSSI_max      4585     21.6551   16.7032
              PCell_RSRP_max      4585     14.1644   10.6029
                   speed_kmh      4611     13.5240    7.3168
                 PCell_SNR_2      4585     11.6296    8.6377
                 PCell_SNR_1      4585      9.0888    6.8214


,feature,n_masked,RMSE,MAE
2,jitter,4514,249925.984598,5459.066079
30,PCell_Uplink_frequency,4420,1079.369788,818.223303
29,PCell_Downlink_frequency,4420,1077.904253,817.943213
34,PCell_freq_MHz,4584,276.970176,171.007853
28,PCell_Cell_ID,4420,167.478367,119.185294
7,COG,4611,66.148063,30.603774
27,PCell_Uplink_Tx_Power_(dBm),4564,38.810033,22.544471
14,Traffic Distance,4611,24.339944,9.829431
19,PCell_RSSI_max,4585,21.655081,16.703196
17,PCell_RSRP_max,4585,14.164397,10.602876


In [25]:

# Create KNN imputer
knn_imputer = KNNImputer(n_neighbors=5, weights='uniform')


knn_imputer.fit(train_scaled)

# Transform validation (on scaled data)
print("  Imputing validation set...")
val_knn_scaled = knn_imputer.transform(val_scaled)

# Inverse transform back to original scale
val_knn_original = scaler.inverse_transform(val_knn_scaled)

# Create full dataframe
val_knn = val_data_masked.copy()
val_knn[features_to_impute] = val_knn_original

# Transform test (on scaled data)
print("  Imputing test set...")
test_knn_scaled = knn_imputer.transform(test_scaled)

# Inverse transform back to original scale
test_knn_original = scaler.inverse_transform(test_knn_scaled)

# Create full dataframe
test_knn = test_data_masked.copy()
test_knn[features_to_impute] = test_knn_original


  Imputing validation set...
  Imputing test set...


In [26]:

method_name = 'KNN Imputation'


# Validation
val_results, val_rmse, val_mae = evaluate_imputation(
    val_knn, val_ground_truth, val_mask_indicator,
    method_name, 'Validation'
)

# Test
test_results, test_rmse, test_mae = evaluate_imputation(
    test_knn, test_ground_truth, test_mask_indicator,
    method_name, 'Test'
)

print(f"\n  VALIDATION:")
print(f"      Overall RMSE: {val_rmse:.4f}")
print(f"      Overall MAE:  {val_mae:.4f}")
print(f"      Features evaluated: {len(val_results)}")

print(f"\n  TEST:")
print(f"      Overall RMSE: {test_rmse:.4f}")
print(f"      Overall MAE:  {test_mae:.4f}")
print(f"      Features evaluated: {len(test_results)}")
    


  VALIDATION:
      Overall RMSE: 39827.8522
      Overall MAE:  142.2052
      Features evaluated: 39

  TEST:
      Overall RMSE: 22720.5874
      Overall MAE:  94.4895
      Features evaluated: 39


In [27]:
val_results, val_rmse, val_mae = evaluate_imputation(
    val_ffill, val_ground_truth, val_mask_indicator, "KNN Imputation", "Validation"
)

represent_columnwise_results(
    val_results,
    split_name="Validation",
    method_name="KNN Imputation",
    sort_by="RMSE",
    ascending=False,
    top_n=20
)



Column-wise Results | Split: Validation | Method: KNN Imputation
-----------------------------------------------------------------
                     feature  n_masked        RMSE       MAE
                      jitter      4514 249925.9846 5459.0661
      PCell_Uplink_frequency      4420   1079.3698  818.2233
    PCell_Downlink_frequency      4420   1077.9043  817.9432
              PCell_freq_MHz      4584    276.9702  171.0079
               PCell_Cell_ID      4420    167.4784  119.1853
                         COG      4611     66.1481   30.6038
 PCell_Uplink_Tx_Power_(dBm)      4564     38.8100   22.5445
            Traffic Distance      4611     24.3399    9.8294
              PCell_RSSI_max      4585     21.6551   16.7032
              PCell_RSRP_max      4585     14.1644   10.6029
                   speed_kmh      4611     13.5240    7.3168
                 PCell_SNR_2      4585     11.6296    8.6377
                 PCell_SNR_1      4585      9.0888    6.8214
  PCell_Downli

,feature,n_masked,RMSE,MAE
2,jitter,4514,249925.984598,5459.066079
30,PCell_Uplink_frequency,4420,1079.369788,818.223303
29,PCell_Downlink_frequency,4420,1077.904253,817.943213
34,PCell_freq_MHz,4584,276.970176,171.007853
28,PCell_Cell_ID,4420,167.478367,119.185294
7,COG,4611,66.148063,30.603774
27,PCell_Uplink_Tx_Power_(dBm),4564,38.810033,22.544471
14,Traffic Distance,4611,24.339944,9.829431
19,PCell_RSSI_max,4585,21.655081,16.703196
17,PCell_RSRP_max,4585,14.164397,10.602876


In [28]:
test_results, test_rmse, test_mae = evaluate_imputation(
    test_ffill, test_ground_truth, test_mask_indicator, "KNN Imputation", "Test"
)

represent_columnwise_results(
    test_results,
    split_name="Test",
    method_name="KNN Imputation",
    sort_by="RMSE",
    ascending=False,
    top_n=50
)



Column-wise Results | Split: Test | Method: KNN Imputation
-----------------------------------------------------------
                     feature  n_masked        RMSE       MAE
                      jitter      4555 173202.1642 4569.3659
    PCell_Downlink_frequency      4411   1112.2258  870.0061
      PCell_Uplink_frequency      4411   1093.5101  852.8991
              PCell_freq_MHz      4604    258.5086  173.4796
               PCell_Cell_ID      4411    157.6062  105.4214
 PCell_Uplink_Tx_Power_(dBm)      4591     40.8126   22.5030
                         COG      4611     22.3376    6.4039
              PCell_RSRP_max      4604     12.6568    8.3745
              PCell_RSSI_max      4604     12.0162    8.0572
                 PCell_SNR_1      4604      8.6281    6.0920
                 PCell_SNR_2      4604      8.4548    5.9146
  PCell_Downlink_Average_MCS      4606      6.9338    5.0211
            Traffic Distance      4611      5.6081    3.0415
                   speed_k

,feature,n_masked,RMSE,MAE
2,jitter,4555,173202.164155,4569.365896
29,PCell_Downlink_frequency,4411,1112.225761,870.006121
30,PCell_Uplink_frequency,4411,1093.510065,852.899116
34,PCell_freq_MHz,4604,258.508554,173.479583
28,PCell_Cell_ID,4411,157.606170,105.421446
27,PCell_Uplink_Tx_Power_(dBm),4591,40.812553,22.502981
7,COG,4611,22.337574,6.403904
17,PCell_RSRP_max,4604,12.656754,8.374464
19,PCell_RSSI_max,4604,12.016227,8.057239
20,PCell_SNR_1,4604,8.628134,6.092033


In [29]:

# Store all results
all_results = {}

# Methods to evaluate
methods = [
    ('Forward/Backward Fill', val_ffill, test_ffill),
    ('KNN (k=5)', val_knn, test_knn)
]

for method_name, val_imputed, test_imputed in methods:
    print(f"EVALUATING: {method_name}")
    
    # Validation
    val_results, val_rmse, val_mae = evaluate_imputation(
        val_imputed, val_ground_truth, val_mask_indicator,
        method_name, 'Validation'
    )
    
    # Test
    test_results, test_rmse, test_mae = evaluate_imputation(
        test_imputed, test_ground_truth, test_mask_indicator,
        method_name, 'Test'
    )
    
    print(f"\n  VALIDATION:")
    print(f"      Overall RMSE: {val_rmse:.4f}")
    print(f"      Overall MAE:  {val_mae:.4f}")
    print(f"      Features evaluated: {len(val_results)}")
    
    print(f"\n  TEST:")
    print(f"      Overall RMSE: {test_rmse:.4f}")
    print(f"      Overall MAE:  {test_mae:.4f}")
    print(f"      Features evaluated: {len(test_results)}")
    
    # Store results
    all_results[method_name] = {
        'val_results': val_results,
        'test_results': test_results,
        'val_rmse': val_rmse,
        'val_mae': val_mae,
        'test_rmse': test_rmse,
        'test_mae': test_mae
    }

EVALUATING: Forward/Backward Fill

  VALIDATION:
      Overall RMSE: 39849.6275
      Overall MAE:  190.5827
      Features evaluated: 39

  TEST:
      Overall RMSE: 27849.2511
      Overall MAE:  170.9081
      Features evaluated: 39
EVALUATING: KNN (k=5)

  VALIDATION:
      Overall RMSE: 39827.8522
      Overall MAE:  142.2052
      Features evaluated: 39

  TEST:
      Overall RMSE: 22720.5874
      Overall MAE:  94.4895
      Features evaluated: 39
